The following exercises are meant to be solved by gathering the bash commands incrimentally in two scripts, one for ex 1.* the other for ex 2.* 

### Ex 1

1\.a Make a new directory called `students` in your home. Download a csv file with the list of students of this lab from [here](https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv) (use the `wget` command) and copy that to `students`. First check whether the file is already there

In [ ]:
cd $HOME
mkdir students #crea la cartella
wget https://www.dropbox.com/scl/fi/bxv17nrbrl83vw6qrkiu9/LCP_22-23_students.csv?rlkey=47fakvatrtif3q3qw4q97p5b7&e=1 #scarica il file
find . -name 'LCP_22-23_students.csv?rlkey=47fakvatrtif3q3qw4q97p5b7' #controlla dov'è il file
cp LCP_22-23_students.csv?rlkey=47fakvatrtif3q3qw4q97p5b7 ./students #copia il file in students
cd students

1\.b Make two new files, one containing the students belonging to PoD, the other to Physics.

In [ ]:
touch PoD_students
touch Physics_students
grep "PoD" LCP_22-23_students.csv?rlkey=47fakvatrtif3q3qw4q97p5b7 >> PoD_students
grep "Physics" LCP_22-23_students.csv?rlkey=47fakvatrtif3q3qw4q97p5b7 >> Physics_students

1\.c For each letter of the alphabet, count the number of students whose surname starts with that letter. 

In [ ]:
touch number_letters
touch just_names
grep -v "Family" LCP_22-23_students.csv?rlkey=47fakvatrtif3q3qw4q97p5b7 > just_names
for i in {A..Z}; do grep -c "^$i" LCP_22-23_students.csv?rlkey=47fakvatrtif3q3qw4q97p5b7 >> number_letters; done #DEVO RIMUOVERE LA PRIMA RIGA!

1\.d Find out which is the letter with most counts.

In [ ]:
paste <(printf "%s\n" {A..Z}) number_letters | sort -k2,2nr | head -1 
# paste <(printf "%s\n" {A..Z}) number_letters = Incolla le lettere in ordine alfabetico all'inizio di ogni riga
# sort -k2,2nr = ordina in ordine numerico (-n) e decrescente (-r) guardando la seconda colonna
# head -1 = prende la prima riga del file

1\.e Assume an obvious numbering of the students in the file (first line is 1, second line is 2, etc.), group students "modulo 18", i.e. 1,19,37,.. 2,20,38,.. etc. and put each group in a separate file  

In [ ]:
paste <(printf "%s\n" {1..74}) just_names > temp && mv temp just_names 
# writes the numbers from 1 to 74 at the beggining of each line.
# I can't do just_names > just_names, I need a temporary variable as temp.
for i in $(seq 1 18); do sed -n "${i}~18p" just_names > group_${i}.txt; done
# per i che va da 1 a 18 stampa le righe i, i+18, i+36,... di just_names sui files group_{i}.
# -n è necessario per evitare la scrittura automatica di "sed", che altrimenti scriverebbe
# tutte le righe a cui va incontro, con -n scrive solo quelle che gli vengono richieste.

### Ex 2

2.a Make a copy of the file `data.csv` removing the metadata and the commas between numbers; call it `data.txt`

In [ ]:
 grep -v "^#" data.csv | sed -e "s/,/ /g" > data.txt
#remove the comments line from data.csv, '|' gives the output of the command before as 
# input of what comes next (sed), sed -e "s/,/ /g": s means substitute 
#g means global, it erase the commas from all the input text


2\.b How many even numbers are there?

In [ ]:
count == 0; while read line; do for n in $line; do ((n%2==0)) && ((count++)); done; done < data.txt; echo $count
# 'awk' is usefull to separate each column, consider a column separated by a blanck space
# NF is the number of fields (columns) in the current line

2\.c Distinguish the entries on the basis of `sqrt(X^2 + Y^2 + Z^2)` is greater or smaller than `100*sqrt(3)/2`. Count the entries of each of the two groups 

In [ ]:
threshold=$(echo "100*sqrt(3)/2" | bc -l); #$(...) executes the command inside the parenthesis: echo "100*sqrt(3)/2" | bc -l gives the expression "..."
# to bc -l that can execute algebra (even sqrt), $((...)) executes simple algebra with just integers (+, -, *, / (divisione intera), %, ++).
while read X Y Z rest;#reads dividing by element each raw. Ex: raw = 1 2 3 4 5 6 => X="1", Y="2", Z="3", rest="4 5 6"
do sum=$(echo "sqrt($X*$X+$Y*$Y+$Z*$Z)" | bc -l) && echo "$sum"; #echo "$sum" = echo $sum
if (( $(echo "$sum > $threshold" | bc -l) )); # we can't use if [$sum -gt $threshold] because this form is just for integer comparison, so we need bc 
# to compare decimal numbers
then echo "$X $Y $Z $rest" >> greater.txt; 
else echo "$X $Y $Z $rest" >> smaller.txt; 
fi; 
done<data.txt

grep -c ".*" greater.txt #count the lines of greater.txt ".*" permitts to count every line (empty or full)
grep -c ".*" smaller.txt



2\.d Make `n` copies of data.txt (with `n` an input parameter of the script), where the i-th copy has all the numbers divided by i (with `1<=i<=n`).

In [ ]:
read -p "Set n: " n; 
while read line; 
do for ((i=1; i<=n; i++)); 
do new_line=""; 
for m in $line; 
do division=$(echo "scale=2; $m/$i" | bc -l) && new_line="$new_line $division"; 
done; 
echo "$new_line" >> "numbers_divided_by_${i}.txt"; 
done; 
done < data.txt